In [1]:
# If any of these fail, run again; Colab sometimes needs a second try
!pip install nltk wordcloud textblob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier

import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

# For Neural Network
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

nltk.download('punkt')
nltk.download('stopwords')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [6]:
from google.colab import files

print("👉 Select dreaddit-train.csv")
uploaded = files.upload()

print("👉 Select dreaddit-test.csv")
uploaded = files.upload()



👉 Select dreaddit-train.csv


Saving dreaddit-train.csv to dreaddit-train.csv
👉 Select dreaddit-test.csv


Saving dreaddit-test.csv to dreaddit-test.csv


In [7]:
train = pd.read_csv("dreaddit-train.csv")
test = pd.read_csv("dreaddit-test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

full = pd.concat([train, test], ignore_index=True)

print("Full shape:", full.shape)
print(full.columns)


Train shape: (2838, 116)
Test shape: (715, 116)
Full shape: (3553, 116)
Index(['subreddit', 'post_id', 'sentence_range', 'text', 'id', 'label',
       'confidence', 'social_timestamp', 'social_karma', 'syntax_ari',
       ...
       'lex_dal_min_pleasantness', 'lex_dal_min_activation',
       'lex_dal_min_imagery', 'lex_dal_avg_activation', 'lex_dal_avg_imagery',
       'lex_dal_avg_pleasantness', 'social_upvote_ratio',
       'social_num_comments', 'syntax_fk_grade', 'sentiment'],
      dtype='object', length=116)


In [8]:
# Define which columns are text / label / subreddit
TEXT_COL = "text"
LABEL_COL = "label"
SUBREDDIT_COL = "subreddit"

# Use all columns EXCEPT raw text and label as features
feature_cols = [c for c in full.columns if c not in [TEXT_COL, LABEL_COL]]
X = full[feature_cols].copy()
y = full[LABEL_COL].values

print("Number of feature columns:", len(feature_cols))
print("X shape:", X.shape)
print("y shape:", y.shape)
print("First 5 feature columns:", feature_cols[:5])


Number of feature columns: 114
X shape: (3553, 114)
y shape: (3553,)
First 5 feature columns: ['subreddit', 'post_id', 'sentence_range', 'id', 'confidence']


In [9]:
# See which columns are non-numeric (object type)
non_numeric_cols = X.select_dtypes(include=["object"]).columns.tolist()
print("Non-numeric columns:", non_numeric_cols)

Non-numeric columns: ['subreddit', 'post_id', 'sentence_range']


In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Categorical feature columns
categorical_cols = ['subreddit', 'post_id', 'sentence_range']

# Find their column indices inside X
cat_indices = [X.columns.get_loc(c) for c in categorical_cols]

# Build column transformer
ct = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_indices)
    ],
    remainder="passthrough"
)

# Apply encoding
X_encoded = ct.fit_transform(X)

print("X_encoded shape:", X_encoded.shape)

from sklearn.preprocessing import StandardScaler

# Convert encoded sparse matrix to dense
X_dense = X_encoded.toarray()

# Scale the dense numeric array
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_dense)

print("Scaled feature shape:", X_scaled.shape)



X_encoded shape: (3553, 3244)
Scaled feature shape: (3553, 3244)


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape, y_train.shape)
print("Testing set :", X_test.shape, y_test.shape)


Training set: (2842, 3244) (2842,)
Testing set : (711, 3244) (711,)


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

log_reg = LogisticRegression(max_iter=2000)  # high max_iter because features are many
log_reg.fit(X_train, y_train)

y_pred_log = log_reg.predict(X_test)

print("===== Logistic Regression =====")
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_log))
print("\nClassification Report:\n", classification_report(y_test, y_pred_log))


===== Logistic Regression =====
Accuracy: 0.7355836849507735

Confusion Matrix:
 [[234 105]
 [ 83 289]]

Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.69      0.71       339
           1       0.73      0.78      0.75       372

    accuracy                           0.74       711
   macro avg       0.74      0.73      0.73       711
weighted avg       0.74      0.74      0.73       711



In [13]:
from sklearn.naive_bayes import GaussianNB

# GaussianNB does NOT accept sparse matrices — convert to dense
X_train_dense = X_train.toarray()
X_test_dense = X_test.toarray()

nb = GaussianNB()
nb.fit(X_train_dense, y_train)

y_pred_nb = nb.predict(X_test_dense)

print("===== Naive Bayes =====")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_nb))


===== Naive Bayes =====
Accuracy: 0.5358649789029536

Confusion Matrix:
 [[ 75 264]
 [ 66 306]]

Classification Report:
               precision    recall  f1-score   support

           0       0.53      0.22      0.31       339
           1       0.54      0.82      0.65       372

    accuracy                           0.54       711
   macro avg       0.53      0.52      0.48       711
weighted avg       0.53      0.54      0.49       711



In [14]:
from sklearn.svm import LinearSVC

svm_clf = LinearSVC(max_iter=5000)
svm_clf.fit(X_train, y_train)

y_pred_svm = svm_clf.predict(X_test)

print("===== Support Vector Machine =====")
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_svm))
print("\nClassification Report:\n", classification_report(y_test, y_pred_svm))


===== Support Vector Machine =====
Accuracy: 0.4767932489451477

Confusion Matrix:
 [[339   0]
 [372   0]]

Classification Report:
               precision    recall  f1-score   support

           0       0.48      1.00      0.65       339
           1       0.00      0.00      0.00       372

    accuracy                           0.48       711
   macro avg       0.24      0.50      0.32       711
weighted avg       0.23      0.48      0.31       711



/usr/local/lib/python3.12/dist-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero

In [15]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

print("===== Decision Tree =====")
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_dt))
print("\nClassification Report:\n", classification_report(y_test, y_pred_dt))


===== Decision Tree =====
Accuracy: 0.6624472573839663

Confusion Matrix:
 [[229 110]
 [130 242]]

Classification Report:
               precision    recall  f1-score   support

           0       0.64      0.68      0.66       339
           1       0.69      0.65      0.67       372

    accuracy                           0.66       711
   macro avg       0.66      0.66      0.66       711
weighted avg       0.66      0.66      0.66       711



In [16]:
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(n_estimators=200, random_state=42)
ada.fit(X_train, y_train)

y_pred_ada = ada.predict(X_test)

print("===== AdaBoost Classifier =====")
print("Accuracy:", accuracy_score(y_test, y_pred_ada))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_ada))
print("\nClassification Report:\n", classification_report(y_test, y_pred_ada))


===== AdaBoost Classifier =====
Accuracy: 0.759493670886076

Confusion Matrix:
 [[257  82]
 [ 89 283]]

Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.76      0.75       339
           1       0.78      0.76      0.77       372

    accuracy                           0.76       711
   macro avg       0.76      0.76      0.76       711
weighted avg       0.76      0.76      0.76       711



In [19]:
# Use the scaled features
X_train_scaled, X_test_scaled, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

input_dim = X_train_scaled.shape[1]

nn2 = Sequential([
    Dense(256, activation='relu', input_shape=(input_dim,)),
    Dropout(0.5),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

nn2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

nn2.summary()

history2 = nn2.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=64,
    verbose=1
)

loss2, acc_nn2 = nn2.evaluate(X_test_scaled, y_test, verbose=0)

print("\n===== Neural Network (Scaled Features) =====")
print("Test Accuracy:", acc_nn2)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 256)            │       830,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 863,745 (3.29 MB)

 Trainable params: 863,745 (3.29 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 5s 72ms/step - accuracy: 0.5471 - loss: 0.9117 - val_accuracy: 0.6872 - val_loss: 0.5744
Epoch 2/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7653 - loss: 0.4862 - val_accuracy: 0.7258 - val_loss: 0.5433
Epoch 3/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8700 - loss: 0.2955 - val_accuracy: 0.7206 - val_loss: 0.5904
Epoch 4/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9495 - loss: 0.1535 - val_accuracy: 0.7135 - val_loss: 0.6612
Epoch 5/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9642 - loss: 0.1067 - val_accuracy: 0.6924 - val_loss: 0.7891
Epoch 6/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9823 - loss: 0.0577 - val_accuracy: 0.6924 - val_loss: 0.8675
Epoch 7/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9895 - loss: 0.0369 - val_accuracy: 0.6889 - val_loss: 0.9299
Epoch 8/15
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.9886 - loss: 0.0358 - val_accuracy: 0.6889 - v